# Week 4 Activity: Delay, Filters, Reverb

Complete this activity as part of your participation grade. Pending length of the lecture, you will have time in class to work. Everything you need to complete this activity can be found in this week's (or a previous week's) lecture code. For this activity, you may want to consult your Audio Tech I notes (or the Computer Music Tutorial)

In [35]:
import numpy as np
from scipy.io.wavfile import read
from scipy.signal import sawtooth, square, butter, filtfilt
import matplotlib.pyplot as plt
from IPython.display import Audio, Image

## Delay
1) Create a function that will modify a passed signal by adding a delay of $m$ milliseconds to the signal.

2) Modify the function such that you can optionally scale the amplitude of the delayed signal 

3) Modify the function so that the user can specify the number of delays to add to the original signal (and optionally, scale each subsequent delay amplitude by a factor of $1/n$

## Filters
1. Create a function that will apply an feedforward comb filter by computing the delay length based off a given resonant frequency in Hz.

In [33]:
def FFCF(x, freq, g=0.8, fs=44100):
    delay_s = 1/freq
    delay_in_samples = int(fs * delay_s)

    y = np.zeros_like(x)
    for n in range(len(x)):
        y[n] = x[n]
        if n - delay_in_samples >= 0:
            y[n] += g * x[n - delay_in_samples]

    return y

(fs, sig) = read("../audio/80spopDrums.wav")

feedback = FFCF(sig, 100)
Audio(feedback, rate = fs)
#plt.magnitude_spectrum(feedback, Fs=fs)
#plt.xlim([0,5000])

2. Create a function that will apply an feedback comb filter by computing the delay length based off a given resonant frequency in Hz.

In [34]:
def FFCF(x, freq, g=0.8, fs=44100):
    delay_s = 1/freq
    delay_in_samples = int(fs * delay_s)

    y = np.zeros_like(x)
    for n in range(len(x)):
        y[n] = x[n]
        if n - delay_in_samples >= 0:
            y[n] += g * y[n - delay_in_samples]

    return y
(fs, sig) = read("../audio/80spopDrums.wav")

feedback = FFCF(sig, 100)
Audio(feedback, rate = fs)
#plt.magnitude_spectrum(feedback, Fs=fs)
#plt.xlim([0,5000])

4. Use your comb filter functions on a wave from the audio folder. Try applying different resonant frequencies and delay lengths. How are the filter results different?

3. Create a function that will apply a butterworth filter to a signal with filter type options 'highpass', 'lowpass', 'bandpass', and 'bandstop'.

## Reverb/Convolution

1. Create a function that will apply a simple moving average filter by convolving the filter kernel and an incoming signal.

2. Apply your filter to a noise signal. What is the effect? What happens if you increase or decrease the kernel size?

3. Create a function that applies convolution reverb to an input signal given an impulse response (this can be default loaded from the audio files). Use np.convolve to create this function.

4. Create another function that applies convolution reverb to an input signal given an impulse response, but this time do not use np.convolve. You should write the convolution from scratch.

    Challenge yourself to create the most efficient function and time your implementation against np.convolve. 
    While developing and testing your function, do not use real audio files. Start with short signals (e.g., impulses, noise, or short sinusoids). Using long signals with loop-based implementations will result in extremely slow run times.

    **Hint:** in a similar manner to how you may have designed your delay function, recall the functions `numpy.zeros` and `numpy.roll` along with how to manipulate multidimensional numpy arrays (e.g., scalar product, summing columns with `vstack`, etc.) If you plan to try `numpy.roll` DO NOT use it in a loop for convolution with a real audio file! (You'll kill your memory), instead consider the `map` function. You may also want to check out the following function which is similar to numpy.roll but more efficient for this task: `scipy.linalg.circulant`.

    You may wish to visit [here](https://numpy.org/doc/stable/user/basics.broadcasting.html) for review of broadcasting (i.e., form some calculation across index value I and column value C) in numpy

In [47]:
def convolver(x,y):
    output = np.zeros(len(x)+len(y)-1)
    for n in range(len(x)):
        start = np.zeros(n)
        g = x[n] * y
        end = np.array([])
        if len(output)-len(g)-n > 0:   
            end = np.zeros(len(output)-len(g)-n)
        output += np.concatenate([start,g,end])[:len(output)]
    return output

x = np.array([2,4,3,6])
y = np.array([1,5,2,3,4])
test = convolver(x,y)
test


array([ 2., 14., 27., 35., 56., 37., 30., 24.])